# Advanced Filtering, Boolean Indexing, and np.where()

In programming, we constantly filter arrays. In TypeScript, you might do this:
```typescript
const adults = ages.filter(age => age >= 18);
```
In NumPy, we use **Boolean Indexing** (sometimes called Boolean Masking). Instead of a loop or filter function, you write a logical expression directly inside the square brackets: `ages[ages >= 18]`.

This works because comparison operators in NumPy are vectorized. Writing `ages >= 18` returns an array of `True`/`False` values (a **boolean mask**). When you pass this mask back into square brackets, NumPy keeps only the elements that align with a `True` value.

There are two major advanced things we will learn in this module:
1.  **Shape-Preserving Filtering**: Standard boolean indexing **flattens** your array. If you filter a 2D matrix, the result is always a 1D array. Why? Because different rows might have a different number of matches, making a grid shape impossible. To keep the original shape, we use `np.where()`, which allows us to replace non-matching elements with a default value (like `0` or `-1`).
2.  **Compound Conditions**: How to combine conditions (e.g., finding people between 18 and 65). We will learn why we *cannot* use standard Python keywords `and` and `or`, and why we must use bitwise operators `&` and `|` instead.

### Simple Explanation with an Everyday Analogy
Imagine a classroom of students seated in a grid of 4 rows and 4 columns. Some students are wearing hats.
*   **Boolean Indexing (Flattens)**: You ask everyone wearing a hat to stand up, leave their desks, and form a single straight line at the front of the class. The original grid layout is lost. You only have a 1D list of students with hats.
*   **The `np.where()` Function (Shape-Preserving)**: You ask everyone who is *not* wearing a hat to temporarily put a gray box over their desk. The students wearing hats stay exactly in their original seats. The grid layout is perfectly preserved!


#### Boolean Indexing Basics
Let's see how a boolean mask is created and how it is used to filter.

In [1]:
import numpy as np

ages = np.array([15, 22, 17, 35, 12, 40])

# 1. Create a boolean mask
mask = ages >= 18
print("Boolean Mask:", mask)

# 2. Apply the mask to filter
adults = ages[mask]
print("Filtered Array (ages[mask]):", adults)

# 3. Doing it in a single line (recommended)
print("Filtered in one line:", ages[ages >= 18])

Boolean Mask: [False  True False  True False  True]
Filtered Array (ages[mask]): [22 35 40]
Filtered in one line: [22 35 40]



#### Compound Conditions (AND / OR)
In NumPy, we use:
*   `&` instead of `and`
*   `|` instead of `or`
*   `~` instead of `not`

*Crucial rule*: You **must** wrap each condition in parentheses `()`. If you do not, Python will fail because bitwise operators like `&` have higher math precedence than comparison operators like `>=`.

In [2]:
import numpy as np

ages = np.array([15, 22, 17, 35, 12, 40])

# Find ages between 18 and 35 (inclusive)
# Note the parentheses!
young_adults = ages[(ages >= 18) & (ages <= 35)]
print("Ages between 18 and 35:", young_adults)

# Find ages that are either under 16 OR over 35
extreme_ages = ages[(ages < 16) | (ages > 35)]
print("Ages under 16 or over 35:", extreme_ages)

Ages between 18 and 35: [22 35]
Ages under 16 or over 35: [15 12 40]


#### Shape-Preserving Filtering with `np.where()`
The syntax is `np.where(condition, value_if_true, value_if_false)`.
If you pass the original array in `value_if_true`, it will keep the matching elements exactly where they were and only replace the non-matching ones.


In [3]:
import numpy as np

grid = np.array([
    [12, 45, 68],
    [99, 15, 23]
])

# Filter any number below 30.
# If it is below 30, replace it with 0. Otherwise, keep it!
filtered_grid = np.where(grid >= 30, grid, 0)

print("Original grid:\n", grid)
print("Filtered grid (shape-preserved):\n", filtered_grid)

Original grid:
 [[12 45 68]
 [99 15 23]]
Filtered grid (shape-preserved):
 [[ 0 45 68]
 [99  0  0]]


### Common Mistakes Beginners Make
1.  **Using `and` or `or` keywords**: Writing `ages[(ages > 18) and (ages < 35)]` will crash! Plain Python `and` tries to evaluate the truth value of the whole array, which triggers a value error: `"The truth value of an array with more than one element is ambiguous"`. Always use `&` and `|`.
2.  **Forgetting parentheses around conditions**: Writing `ages[ages >= 18 & ages <= 35]` will cause a crash or weird behavior. Without parentheses, Python evaluates `18 & ages` first because `&` is a bitwise operator and has a higher priority than `>=`. Always write `(ages >= 18) & (ages <= 35)`.
3.  **Forgetting that Boolean Indexing flattens 2D arrays**: If you do `matrix[matrix > 5]`, you get a 1D array. If you need a 2D matrix back, you must use `np.where()`.



#### Exercise 1
You have a list of numbers:
```python
nums = np.array([3, -1, 4, -5, 0, 9, -2])
```
1. Use boolean indexing to filter out and keep only the positive numbers (numbers greater than `0`).
2. Filter and keep only the negative numbers.


In [4]:
import numpy as np
nums = np.array([3, -1, 4, -5, 0, 9, -2])

positives = nums[nums > 0]
negatives = nums[nums < 0]

print("Positives:", positives)
print("Negatives:", negatives)

Positives: [3 4 9]
Negatives: [-1 -5 -2]


*Explanation*: `nums > 0` returns `[True, False, True, False, False, True, False]`, which selects elements at indices 0, 2, and 5.


#### Exercise 2 (Medium)
You are working with student scores in a 2D grid:
```python
scores = np.array([
    [85, 42, 90],
    [60, 31, 75],
    [95, 88, 55]
])
```
1. Get a flat 1D array of all scores that are passing (scores `>= 60`).
2. Use `np.where()` to create a new 2D array where all failing scores (under `60`) are replaced by `-1` (to flag them for re-testing), but passing scores remain unchanged.


In [5]:
import numpy as np
scores = np.array([
    [85, 42, 90],
    [60, 31, 75],
    [95, 88, 55]
])

# 1. Flat list of passing scores
passing = scores[scores >= 60]

# 2. Shape-preserved array with failing flagged as -1
flagged = np.where(scores >= 60, scores, -1)

print("Passing scores (flat):", passing)
print("Flagged scores (2D):\n", flagged)

Passing scores (flat): [85 90 60 75 95 88]
Flagged scores (2D):
 [[85 -1 90]
 [60 -1 75]
 [95 88 -1]]


*Explanation*:
*   `scores[scores >= 60]` returns a 1D array because row 1 has two passes, row 2 has two, and row 3 has two, but row patterns can differ. Boolean indexing always flattens.
*   `np.where(scores >= 60, scores, -1)` keeps the original cell value if the condition is met, and places `-1` if it is not, preserving the 3x3 grid shape.



#### Exercise 3
Given an array of warehouse inventory codes:
```python
codes = np.array([101, 102, 201, 305, 108, 202, 401])
```
We want to extract codes that belong to Department 1 (codes in the `100` range: `100 <= code < 200`) AND are even numbers (divisible by 2 with no remainder).
*Hint: Use the modulo operator `% 2 == 0` for even numbers.*


In [6]:
import numpy as np
codes = np.array([101, 102, 201, 305, 108, 202, 401])

# Condition 1: In the 100 range -> (codes >= 100) & (codes < 200)
# Condition 2: Even -> (codes % 2 == 0)
matching_codes = codes[(codes >= 100) & (codes < 200) & (codes % 2 == 0)]
print("Matching codes:", matching_codes)

Matching codes: [102 108]


*Explanation*: Only `102` and `108` are both in the `100-199` range and are even numbers.